# 01.05 AWS Bedrock

`langchain-aws`는 **AWS Bedrock**을 통한 Claude / Nova / Llama / Mistral / Cohere 통합 접근을 제공한다. 두 가지 진입점이 있다.

- `ChatBedrock` — 공급자별 네이티브 API 포맷(구형 / 세밀 제어)
- `ChatBedrockConverse` — AWS Converse API **통합 포맷**(권장, 모든 공급자 공통)

Amazon Nova(`us.amazon.nova-lite-v1:0` 등)는 **Converse 경유**로만 호출한다.

## 학습 목표

- `ChatBedrockConverse`로 여러 공급자 모델을 같은 인터페이스로 쓴다
- `ChatBedrock`(레거시) 사용 조건을 안다 — provider-specific 기능이 필요할 때만
- Amazon Nova 모델을 Converse 경로로 호출한다
- 지역(`region`)과 크로스-리전 inference profile(`us.<model>:0`) 의미를 구분한다

## 언제 쓰나

- AWS 조직 계정 · IAM · VPC 경계 안에서 모델을 호출해야 할 때
- 한 코드베이스에서 Claude, Nova, Llama, Mistral을 **Converse 하나**로 교체하고 싶을 때

## 01.05.1 환경 설정

`.env`에 `AWS_ACCESS_KEY_ID`, `AWS_SECRET_ACCESS_KEY`(선택 `AWS_SESSION_TOKEN`, `AWS_REGION`)가 필요하다. 해당 리전에서 Bedrock 모델 액세스가 **활성화**되어 있어야 한다.

In [ ]:
# !pip install -U langchain langchain-aws boto3
import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("AWS_ACCESS_KEY_ID"), "AWS_ACCESS_KEY_ID 필요"
assert os.environ.get("AWS_SECRET_ACCESS_KEY"), "AWS_SECRET_ACCESS_KEY 필요"

## 01.05.2 `ChatBedrockConverse` — 권장 경로

Converse API는 AWS가 정의한 공급자-독립 메시지 포맷. 모델 ID만 바꾸면 Claude ↔ Nova ↔ Llama 교체 가능.

In [ ]:
from langchain_aws import ChatBedrockConverse

region = os.environ.get("AWS_REGION", "us-east-1")
claude = ChatBedrockConverse(
    model="anthropic.claude-3-5-sonnet-20241022-v2:0",
    region_name=region,
    max_tokens=512,
)
msg = claude.invoke("Bedrock Converse API의 장점을 한 문장으로 설명해줘.")
print(msg.content)
print("usage:", msg.usage_metadata)

## 01.05.3 Amazon Nova 호출 — Converse 경유

Nova는 Amazon 자체 모델군(lite/pro/micro). **모델 ID는 반드시 `us.amazon.nova-*:0` 형태의 cross-region inference profile** 을 쓴다.

In [ ]:
nova = ChatBedrockConverse(
    model="us.amazon.nova-lite-v1:0",
    region_name=region,
    max_tokens=512,
)
print("nova →", nova.invoke("안녕, 너는 어느 회사의 모델이야?").content[:120])

## 01.05.4 `ChatBedrock` — 레거시 / provider-specific

provider-native 포맷이 필요할 때만 사용. `provider="anthropic"` 등으로 페이로드 형식을 지정한다.

In [ ]:
from langchain_aws import ChatBedrock

legacy = ChatBedrock(
    model="anthropic.claude-3-5-haiku-20241022-v1:0",
    provider="anthropic",
    region=region,
    model_kwargs={"max_tokens": 256, "temperature": 0},
)
print("legacy →", legacy.invoke("한 문장으로 자기소개").content[:120])

## 01.05.5 Tool calling (Converse)

Converse API는 tool calling을 공통 포맷으로 제공한다. 모든 Converse 대응 모델에서 동일 코드로 작동한다.

In [ ]:
from langchain_core.tools import tool

@tool
def sub(a: int, b: int) -> int:
    """두 정수를 뺀다."""
    return a - b

bound = claude.bind_tools([sub])
r = bound.invoke("42에서 17을 빼면?")
print("tool_calls:", r.tool_calls)

## 01.05.6 Structured output

In [ ]:
from pydantic import BaseModel

class Diff(BaseModel):
    a: int
    b: int
    result: int

structured = claude.with_structured_output(Diff)
print(structured.invoke("100에서 7을 빼면 93이야. 구조화해줘."))

## 01.05.7 `init_chat_model()` 경로

In [ ]:
from langchain.chat_models import init_chat_model

m = init_chat_model(
    "bedrock_converse:anthropic.claude-3-5-haiku-20241022-v1:0",
    region_name=region,
    max_tokens=128,
)
print(m.invoke("핑").content[:60])

## 체크리스트

- [ ] `ChatBedrockConverse`를 **기본값**으로 사용하고, `ChatBedrock`은 provider-specific 기능이 필요할 때만 썼다
- [ ] 모델 ID가 해당 리전에서 활성화되어 있고 IAM 권한이 충분한지 확인했다
- [ ] Nova는 `us.amazon.nova-*` cross-region profile을 사용했다
- [ ] `ChatAmazonNova` 클래스는 이 버전의 `langchain-aws`에 **존재하지 않음**(Converse 경유로 호출) 을 기억한다

## 다음

- `02_anthropic.ipynb` — 직접 Anthropic API 호출 경로와 비교
- `11_provider_middleware/06_bedrock_prompt_caching.ipynb` — Bedrock 전용 prompt caching

## 참고

- 공식: https://docs.langchain.com/oss/python/integrations/chat/bedrock
- Converse API: https://docs.aws.amazon.com/bedrock/latest/userguide/conversation-inference.html